In [3]:
import numpy as np
import pandas as pd
import polars as pl
from sentence_transformers import SentenceTransformer

# What this does: load the dataset
df = pl.read_parquet("spy_10k_2015_present.parquet").to_pandas()

# What this does: load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# What this does: generate embeddings from disclosure text
df["embedding"] = model.encode(
    df["esg_text"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True
)

c:\Users\ddddd\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 452.44it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeyError: 'esg_text'

In [2]:
from sklearn.metrics.pairwise import cosine_similarity

X = np.vstack(df["embedding"].values)

similarity_matrix = cosine_similarity(X)

KeyError: 'embedding'

In [1]:
import numpy as np
import pandas as pd
import polars as pl
from sklearn.metrics.pairwise import cosine_similarity

# What this does: load parquet
df = pl.read_parquet("spy_10k_2015_present.parquet").to_pandas()

# What this does: ensure embeddings are numpy arrays
df["embedding"] = df["embedding"].apply(lambda x: np.asarray(x, dtype=float))


def knn_elbow_analysis(
    df,
    embedding_col="embedding",
    sector_col="gics_sector",
    year_col="filing_year",
    max_k=20
):

    results = []

    for (sector, year), grp in df.groupby([sector_col, year_col]):

        if len(grp) < max_k + 1:
            continue

        X = np.vstack(grp[embedding_col].values)

        sim = cosine_similarity(X)

        for k in range(1, max_k + 1):

            sims = []

            for i in range(len(grp)):

                peer_mask = np.ones(len(grp), dtype=bool)
                peer_mask[i] = False

                peer_sims = sim[i, peer_mask]

                topk = np.sort(peer_sims)[-k:]

                sims.append(topk.mean())

            results.append({
                "sector": sector,
                "year": year,
                "k": k,
                "mean_similarity": np.mean(sims)
            })

    return pd.DataFrame(results)


elbow_df = knn_elbow_analysis(df)

KeyError: 'embedding'

In [ ]:
import matplotlib.pyplot as plt

elbow_curve = elbow_df.groupby("k")["mean_similarity"].mean()

plt.figure()

plt.plot(elbow_curve.index, elbow_curve.values)

plt.xlabel("k (Nearest Neighbours)")
plt.ylabel("Average Cosine Similarity")

plt.title("k-NN Elbow Analysis for ESG Disclosure Similarity")

plt.show()

In [ ]:
def detect_elbow(k_vals, y_vals):

    diffs = np.diff(y_vals)
    second_diff = np.diff(diffs)

    elbow_index = np.argmax(np.abs(second_diff)) + 2

    return k_vals[elbow_index]


k_values = elbow_curve.index.values
y_values = elbow_curve.values

optimal_k = detect_elbow(k_values, y_values)

print("Optimal k:", optimal_k)

NameError: name 'semantic_distinctiveness_0_1' is not defined

ESGadj = ESG * (1 - KNN)